# Unidade 5 — Guia de estudo

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/correa-ufrrj/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_05/00_guia_da_unidade.ipynb)

## Do conjunto aos grupos: como revelar propriedades entre subconjuntos de dados?

Na Unidade 4, passamos da observação de registros isolados para a extração de **propriedades coletivas** do conjunto de dados. Frequências, distribuições, relações entre variáveis, coocorrências e representações textuais são exemplos de propriedades que não pertencem a um registro individual: elas aparecem quando os dados são considerados em conjunto.

A novidade desta unidade é introduzir uma nova operação: **agrupar os dados**. Em vez de descrever apenas o conjunto como um todo, dividiremos seus registros segundo critérios explícitos e investigaremos propriedades que aparecem **entre os grupos**.

Um agrupamento não é determinado automaticamente pelo dataset. Podemos organizar os mesmos registros por `local`, `genero`, `tema`, período ou por outro critério coerente com uma pergunta de pesquisa. Assim, antes de perguntar se dois grupos diferem, precisamos perguntar **por que esses grupos foram formados e qual propriedade desejamos comparar entre eles**.

![Dois conjuntos de documentos passam por distribuições e incerteza; dois textos passam por conjuntos, vetores e alinhamento; ambos os caminhos retornam a documentos inspecionados por uma lupa.](https://github.com/correa-ufrrj/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_04/imagens/00_abertura_conceitual.png?raw=1)

*Ilustração conceitual gerada para a abertura. Os documentos são genéricos e fictícios; a imagem não constitui evidência histórica.*

**Problema orientador:** como o agrupamento dos dados permite revelar diferenças e semelhanças que não aparecem quando analisamos apenas o conjunto como um todo?

A unidade seguirá dois percursos complementares. No primeiro, compararemos grupos por propriedades numéricas ou categóricas. No segundo, representaremos documentos e compararemos grupos e documentos por propriedades textuais. Em ambos, a tarefa central será justificar **o critério de agrupamento, a propriedade observada, a medida escolhida e o alcance da interpretação**.

In [1]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/correa-ufrrj/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/dados/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_05'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

Ambiente preparado em: /dados/disciplina_computacao_aplicada_humanidades_digitais/unidade_05


A pergunta orientadora se divide em dois blocos que compartilham a mesma lógica: formar grupos, escolher uma propriedade que possa ser comparada e retornar aos casos para interpretar o resultado.

## Objetivos e percurso

![Seis etapas ligam pergunta, estimativa, incerteza, representação textual, comparação e interpretação; uma seta retorna às decisões anteriores.](https://github.com/correa-ufrrj/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_04/imagens/00_percurso_comparacao.svg?raw=1)

### Bloco 1 — Propriedades numéricas e categóricas entre grupos

Pergunta geral: **em que medida grupos definidos no dataset diferem quanto a uma propriedade numérica ou categórica?**

Nesse bloco, partiremos do critério de agrupamento e distinguiremos estimativa, tamanho de efeito, incerteza, valor de *p* e relevância substantiva. O objetivo não é apenas detectar uma diferença, mas compreender **quanto os grupos diferem, quão estável é essa diferença e o que ela significa no contexto da pergunta**.

### Bloco 2 — Propriedades textuais entre grupos e documentos

Pergunta geral: **em que medida grupos de documentos diferem ou se aproximam quanto às suas propriedades textuais?**

Nesse bloco, construiremos matriz documento-termo e TF-IDF e usaremos medidas de comparação textual. A representação textual será tratada como uma forma de situar os documentos em um espaço comum, permitindo observar proximidades e diferenças que só aparecem quando os documentos são comparados entre si e em relação à coleção.

| Notebook | Pergunta central | Produto |
|---|---|---|
| 00 | Como formar grupos e definir uma comparação defensável? | Escolha preliminar |
| 01 | Qual é o tamanho da diferença entre os grupos? | Quadro de estimativas |
| 02 | Que incerteza acompanha a estimativa? | Ficha inferencial |
| 03 | Como representar documentos para compará-los? | Matriz textual |
| 04 | Em que sentido documentos ou grupos textuais são semelhantes? | Relatório de pares e grupos |
| 05 | Que comparação sustenta o argumento? | Análise comparativa |

Ao final, você deverá distinguir estimativa, incerteza, valor de *p*, tamanho de efeito e relevância; construir matriz documento-termo, TF-IDF e três medidas de comparação textual; e inspecionar qualitativamente os casos.

O princípio comum aos dois blocos é:

$$
\text{dataset} \rightarrow \text{critério de agrupamento} \rightarrow \text{grupos} \rightarrow \text{propriedade} \rightarrow \text{comparação} \rightarrow \text{casos} \rightarrow \text{interpretação}.
$$

In [ ]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

# Mantemos a mesma lógica de geração usada na Unidade 4.
# A semente torna o conjunto inteiramente reprodutível.
SEED = 20261058
N = 120
rng = np.random.default_rng(SEED)

# Seis documentos para cada ano entre 1890 e 1909.
linhas = []
for ano in range(1890, 1910):
    t = (ano - 1890) / 19
    p_capital = 0.12 + 0.76 * t

    # Há documentos dos dois locais em todos os anos.
    n_capital = int(np.clip(rng.binomial(6, p_capital), 1, 5))
    locais_ano = np.array(
        ["Capital"] * n_capital + ["Interior"] * (6 - n_capital)
    )
    rng.shuffle(locais_ano)
    linhas.extend((ano, local) for local in locais_ano)

rng.shuffle(linhas)
anos = np.array([linha[0] for linha in linhas])
locais = np.array([linha[1] for linha in linhas])
capital = locais == "Capital"

# Gênero documental.
generos = rng.choice(
    ["notícia", "editorial", "carta"],
    size=N,
    p=[0.45, 0.30, 0.25],
)

# Tema: as probabilidades dependem do gênero.
temas_possiveis = np.array(["educação", "trabalho", "progresso", "saúde"])
probabilidades_tema = {
    "notícia":   [0.20, 0.25, 0.15, 0.40],
    "editorial": [0.20, 0.30, 0.35, 0.15],
    "carta":     [0.40, 0.25, 0.20, 0.15],
}
temas = np.array([
    rng.choice(temas_possiveis, p=probabilidades_tema[genero])
    for genero in generos
])

# Extensão do documento: gênero, local, período e variação individual contribuem.
efeito_genero = {"notícia": 0.03, "editorial": 0.24, "carta": -0.18}
log_palavras = (
    math.log(650)
    + np.array([efeito_genero[g] for g in generos])
    + np.where(capital, 0.64, 0.0)
    - 0.016 * (anos - 1890)
    + rng.normal(0, 0.24, N)
)
palavras = np.maximum(140, np.rint(np.exp(log_palavras))).astype(int)

# Poucos documentos excepcionalmente longos.
especiais = rng.choice(N, 3, replace=False)
palavras[especiais] = np.rint(
    palavras[especiais] * rng.uniform(1.65, 2.05, 3)
).astype(int)

# Palavras por página variam entre documentos.
densidade_pagina = np.clip(rng.normal(365, 38, N), 270, 470)
paginas = np.maximum(1, np.rint(palavras / densidade_pagina)).astype(int)

# Menções a pessoas dependem da extensão e do gênero.
taxa_pessoas = {"notícia": 7.0, "editorial": 4.7, "carta": 5.8}
media_pessoas = palavras / 1000 * np.array(
    [taxa_pessoas[g] for g in generos]
)
pessoas = rng.poisson(media_pessoas)

documentos = pd.DataFrame({
    "id_documento": [f"R{i:03d}" for i in range(1, N + 1)],
    "ano": anos,
    "genero": generos,
    "local": locais,
    "tema": temas,
    "palavras": palavras,
    "paginas": paginas,
    "pessoas": pessoas,
})

# A própria aula gera os dados que utilizará.
Path("dados").mkdir(exist_ok=True)

documentos.to_csv("dados/documentos.csv", index=False)

proveniencia = {
    "natureza": "dados sintéticos gerados para fins didáticos",
    "semente": SEED,
    "numero_registros": N,
    "unidade_analise": "documento fictício",
    "observacao": (
        "As relações entre as variáveis foram introduzidas deliberadamente "
        "pelo mecanismo gerador. Os dados não constituem evidência histórica."
    ),
}

Path("dados/proveniencia.json").write_text(
    json.dumps(proveniencia, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Arquivo gerado: dados/documentos.csv")
print("Arquivo gerado: dados/proveniencia.json")
print("Registros:", len(documentos))
print("Natureza:", proveniencia["natureza"])

documentos.head(3)


O conjunto acima é gerado pelo próprio notebook, de forma reprodutível, com a mesma lógica básica usada na Unidade 4. Ele contém 120 documentos fictícios. Essa continuidade é intencional: na Unidade 4 examinamos propriedades coletivas do conjunto; agora perguntamos como essas propriedades mudam quando organizamos os registros em grupos. As relações entre as variáveis foram introduzidas deliberadamente pelo mecanismo gerador e não constituem evidência histórica nem uma amostra probabilística.

## Antes de comparar: o que é um grupo?

Chamaremos de **grupo** um subconjunto de registros definido por um critério explícito. Por exemplo, podemos separar os documentos por `local`, `genero`, `tema` ou período. O mesmo dataset pode, portanto, admitir diferentes agrupamentos, e cada um responde a uma pergunta diferente.

Formar grupos não significa afirmar que seus integrantes são homogêneos ou que pertencem a tipos naturais. Uma diferença entre grupos é uma propriedade agregada da comparação segundo determinada variável ou representação. Mesmo que a média de um grupo seja maior que a de outro, os valores individuais podem apresentar grande sobreposição.

Por isso, toda comparação deve explicitar:

1. qual é a unidade de análise;
2. qual critério define os grupos;
3. qual propriedade será comparada;
4. qual medida representa essa propriedade;
5. que heterogeneidade existe dentro dos grupos;
6. que casos qualificam ou desafiam o resultado agregado.

## Regra de interpretação

Para toda comparação, registre:

1. unidades, critério de agrupamento e grupos resultantes;
2. propriedade e medida escolhidas;
3. diferença ou similaridade observada;
4. incerteza ou sensibilidade pertinente;
5. casos que qualificam o agregado;
6. afirmação sustentada e limite.

Intervalos e testes só possuem interpretação inferencial sob um procedimento e uma população-alvo defensáveis. Em um corpus completo, a diferença descritiva pode ser o resultado relevante; em uma coleção de conveniência, um pequeno valor de *p* não repara o mecanismo de seleção.

## Questões críticas

Elas atravessarão todos os experimentos:

1. Por que estes registros foram tratados como grupos distintos?
2. O agrupamento escolhido é relevante para a pergunta de pesquisa?
3. Que propriedade está sendo comparada entre os grupos?
4. Há heterogeneidade e sobreposição dentro dos grupos?
5. Como a escolha da medida altera a comparação?
6. Uma diferença estatística é necessariamente relevante?
7. No caso textual, documentos do mesmo grupo também apresentam perfis textuais semelhantes?

## U05-A01 — Diagnóstico inicial

**Atividade individual — 10 minutos.** Responda em Markdown:

- Escolha duas formas diferentes de agrupar os mesmos registros deste dataset. Que perguntas cada agrupamento permitiria formular?
- Uma média maior em um grupo implica que todos os seus registros têm valores maiores que os do outro grupo?
- Quando uma diferença de médias é substantivamente importante?
- Um intervalo de confiança contém 95% dos dados?
- O valor de *p* informa a probabilidade de a hipótese nula ser verdadeira?
- Dois documentos classificados no mesmo grupo são necessariamente semelhantes?

**Minha resposta:** Escreva aqui.

O diagnóstico torna visíveis duas decisões que acompanharão toda a unidade: **como os grupos são formados** e **qual propriedade será usada para compará-los**. O produto final deverá mostrar que essas escolhas antecedem a aplicação das métricas.

## Produto e avaliação

A análise comparativa será avaliada por alinhamento à pergunta, justificativa do agrupamento, comparabilidade dos grupos, correção da medida, transparência dos pressupostos, inspeção de casos, relevância substantiva, limites e reprodutibilidade.

Em todos os exercícios, evite transformar uma propriedade agregada dos grupos em uma afirmação automática sobre cada indivíduo. O resultado deve ser lido como uma propriedade da comparação realizada segundo determinado critério e determinada medida.

Siga para o Notebook 01. Começaremos por um agrupamento explícito e pelo tamanho observado da diferença entre os grupos antes de introduzir testes ou intervalos.